# 28장 · 네트워크와 센서망 — 끊긴 곳, 빈 곳 찾기

이산위상 연구실 활용 편 · 글 안상선. 그대로 위에서부터 실행하면 돼요.

In [1]:
# 처음 한 번만: 필요한 패키지 설치 (구글 코랩이면 앞에 !를 붙여 그대로 실행)
# pip install numpy scipy matplotlib scikit-image networkx trimesh ripser

센서를 무작위로 놓고 반경을 바꿔 가며 **통신 덩어리 수**(그래프의 연결 성분)와 **음영 지역**(어느 센서도 닿지 않는 곳)을 셉니다. `networkx`를 씁니다.

In [2]:
import numpy as np, networkx as nx
from scipy import ndimage

rng = np.random.default_rng(21)
W, H, n = 320, 220, 16
P = np.column_stack([18 + rng.random(n) * (W - 36), 18 + rng.random(n) * (H - 36)])

def analyze(r, P=P):
    G = nx.Graph(); G.add_nodes_from(range(len(P)))
    for a in range(len(P)):
        for b in range(a + 1, len(P)):
            if np.hypot(*(P[a] - P[b])) <= 2 * r:     # 두 원이 겹치면 통신
                G.add_edge(a, b)
    gx, gy = np.meshgrid(np.arange(3, W, 6), np.arange(3, H, 6))
    d = np.min(np.hypot(gx[..., None] - P[:, 0], gy[..., None] - P[:, 1]), axis=2)
    uncovered = d > r
    lab, k = ndimage.label(uncovered)
    border = set(np.unique(np.concatenate([lab[0], lab[-1], lab[:, 0], lab[:, -1]]))) - {0}
    return nx.number_connected_components(G), k - len(border), uncovered.mean()

for r in [15, 20, 26, 32, 40, 50]:
    c, inner, frac = analyze(r)
    print(f"반경 {r:>2}m: 통신 덩어리 {c:>2}개, 안쪽 음영 구멍 {inner}곳, 빈 넓이 {frac:.0%}")

반경 15m: 통신 덩어리 12개, 안쪽 음영 구멍 0곳, 빈 넓이 85%
반경 20m: 통신 덩어리 11개, 안쪽 음영 구멍 1곳, 빈 넓이 74%
반경 26m: 통신 덩어리  4개, 안쪽 음영 구멍 1곳, 빈 넓이 60%
반경 32m: 통신 덩어리  3개, 안쪽 음영 구멍 3곳, 빈 넓이 47%
반경 40m: 통신 덩어리  2개, 안쪽 음영 구멍 3곳, 빈 넓이 33%
반경 50m: 통신 덩어리  1개, 안쪽 음영 구멍 2곳, 빈 넓이 19%


반경이 작으면 통신이 끊기고 음영이 넓습니다. 「안쪽에 갇힌 음영」은 센서망 속 **구멍**이에요. 안쪽에 갇힌 음영 가운데 가장 큰 곳 한가운데에 센서를 하나 더 놓아 봅니다.

In [3]:
r = 26
gx, gy = np.meshgrid(np.arange(3, W, 6), np.arange(3, H, 6))
d = np.min(np.hypot(gx[..., None] - P[:, 0], gy[..., None] - P[:, 1]), axis=2)
lab, k = ndimage.label(d > r)
border = set(np.unique(np.concatenate([lab[0], lab[-1], lab[:, 0], lab[:, -1]]))) - {0}
sizes = np.bincount(lab.ravel()); sizes[0] = 0
for b in border: sizes[b] = 0                                   # 가장자리 빈 곳 말고 안쪽에 갇힌 구멍만
big = sizes.argmax()
new = np.array([[gx[lab == big].mean(), gy[lab == big].mean()]])
print('추가 전:', analyze(r)[:2], ' 추가 후:', analyze(r, np.vstack([P, new]))[:2])

추가 전: (4, 1)  추가 후: (4, 0)
